In [ ]:
"""
PIPELINE STAGE 2: Study-Level Alignment and Cleaning
----------------------------------------------------
Role in Thesis: 
    Links the unstructured generated findings from LLaVA-Rad back to their 
    original clinical reference reports using strict case identifiers.

Key Verifications (Conservation of Mass):
    - Asserts that N_total = N_aligned + N_unmatched_subset.
    - Explicitly excludes pairs with empty generated/reference text to 
      prevent artifacts in downstream RadGraph processing.
      
Outputs:
    - aligned_inference_{subset}_subset.csv (Valid analytical cohort)
    - empty_report_rows.csv (Failure artifact)
    - unmatched_*_rows.csv (Matching diagnostics)
"""

In [ ]:
# =====================================================================
# DATA ALIGNMENT & MASS CONSERVATION CHECK
# =====================================================================
# Performs an outer join to align generated LLaVA-Rad findings with the 
# clinical reference subset based on the unique 'study_id'.
# We explicitly preserve '_merge' indicators to track unmatched records 
# conceptually (unmatched_subset, unmatched_inference) and prevent silent data loss.

# Validation (Conservation of Mass):
# Ensures that N_total = N_aligned + N_unmatched_subset.
# This proves that the pandas join logic introduced no combinatorial errors.

# =====================================================================
# FILTERING UNUSABLE PAIRS (CLEANING)
# =====================================================================
# Empty reports must be explicitly excluded before downstream analysis. 
# A report with zero textual content cannot yield viable RadGraph entities, 
# resulting in infinite or NaN overlaps. These artifacts are persisted to 
# 'empty_report_rows.csv' for methodological transparency.

# =========================
# 1. Imports
# =========================
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd


# =========================
# 2. Config
# =========================
BASE_DIR = Path("/workspace/thesis")

INFER_JSONL_PATH = BASE_DIR / "outputs_raw" / "main" / "subset_1500_run.jsonl"
SUBSET_PATH = BASE_DIR / "data" / "queries" / "subset_1500.json"   # <-- anpassen
OUT_DIR = BASE_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ALIGNED_CSV_OUT = OUT_DIR / "aligned_inference_1500_subset.csv"
ALIGNED_JSONL_OUT = OUT_DIR / "aligned_inference_1500_subset.jsonl"

RADGRAPH_INPUT_CSV_OUT = OUT_DIR / "radgraph_input_1500_subset.csv"
RADGRAPH_INPUT_JSONL_OUT = OUT_DIR / "radgraph_input_1500_subset.jsonl"

UNMATCHED_INFER_OUT = OUT_DIR / "unmatched_inference_rows.csv"
UNMATCHED_SUBSET_OUT = OUT_DIR / "unmatched_subset_rows.csv"
EMPTY_REPORTS_OUT = OUT_DIR / "empty_report_rows.csv"


# =========================
# 3. Candidate columns
# =========================
# Inference side
GEN_CANDIDATES = [
    "generated_report", "generated_text", "prediction", "pred_report",
    "model_output", "output", "answer", "text", "response"
]

# Common keys
ID_CANDIDATES = ["id", "sample_id", "uid", "example_id"]
STUDY_ID_CANDIDATES = ["study_id", "studyid", "StudyID"]
DICOM_ID_CANDIDATES = ["dicom_id", "dicomid", "DicomID"]
IMAGE_PATH_CANDIDATES = ["image_path", "img_path", "path", "file_path", "image", "image_file"]

# Subset reference side
REFERENCE_REPORT_CANDIDATES = [
    "reference_report", "report", "findings", "reference_findings",
    "ground_truth", "target", "impression"
]

IMPRESSION_CANDIDATES = [
    "impression", "reference_impression"
]

FINDINGS_CANDIDATES = [
    "findings", "reference_findings"
]


# =========================
# 4. Helper functions
# =========================
def load_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".jsonl":
        return pd.read_json(path, lines=True)
    if suffix == ".json":
        return pd.read_json(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported file format: {path}")

def first_existing_column(df: pd.DataFrame, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = x.replace("\n", " ").replace("\r", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x

def normalize_id(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    if x == "":
        return None
    return x

def basename_without_ext(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    if x == "":
        return None
    return Path(x).stem

def print_block(title, value):
    print(f"\n{'='*15} {title} {'='*15}")
    print(value)

def add_key_columns(df, prefix=""):
    df = df.copy()

    id_col = first_existing_column(df, ID_CANDIDATES)
    study_col = first_existing_column(df, STUDY_ID_CANDIDATES)
    dicom_col = first_existing_column(df, DICOM_ID_CANDIDATES)
    image_col = first_existing_column(df, IMAGE_PATH_CANDIDATES)

    df[f"{prefix}id_key"] = df[id_col].map(normalize_id) if id_col else None
    df[f"{prefix}study_id_key"] = df[study_col].map(normalize_id) if study_col else None
    df[f"{prefix}dicom_id_key"] = df[dicom_col].map(normalize_id) if dicom_col else None
    df[f"{prefix}image_path_key"] = df[image_col].map(normalize_id) if image_col else None
    df[f"{prefix}image_stem_key"] = df[image_col].map(basename_without_ext) if image_col else None

    meta = {
        "id_col": id_col,
        "study_col": study_col,
        "dicom_col": dicom_col,
        "image_col": image_col
    }
    return df, meta

def build_reference_report(df):
    report_col = first_existing_column(df, REFERENCE_REPORT_CANDIDATES)
    findings_col = first_existing_column(df, FINDINGS_CANDIDATES)
    impression_col = first_existing_column(df, IMPRESSION_CANDIDATES)

    if report_col:
        return df[report_col].map(normalize_text), report_col

    if findings_col and impression_col:
        ref = (
            df[findings_col].fillna("").map(normalize_text) + " " +
            df[impression_col].fillna("").map(normalize_text)
        ).str.strip()
        ref = ref.map(normalize_text)
        return ref, f"{findings_col}+{impression_col}"

    if findings_col:
        return df[findings_col].map(normalize_text), findings_col

    if impression_col:
        return df[impression_col].map(normalize_text), impression_col

    raise ValueError("No reference report column found on subset side.")

def build_generated_report(df):
    gen_col = first_existing_column(df, GEN_CANDIDATES)
    if gen_col is None:
        raise ValueError(f"No generated report column found. Tried: {GEN_CANDIDATES}")
    return df[gen_col].map(normalize_text), gen_col

def choose_best_value(row, cols):
    for c in cols:
        if c in row and pd.notna(row[c]) and str(row[c]).strip() != "":
            return row[c]
    return None

def safe_case_id(row):
    for c in ["id", "study_id", "dicom_id", "image_stem", "image_path"]:
        if c in row and pd.notna(row[c]) and str(row[c]).strip() != "":
            return str(row[c])
    return str(row.name)


# =========================
# 5. Load data
# =========================
infer_df = load_table(INFER_JSONL_PATH)
subset_df = load_table(SUBSET_PATH)

print_block("Inference shape", infer_df.shape)
print_block("Subset shape", subset_df.shape)
print_block("Inference columns", list(infer_df.columns))
print_block("Subset columns", list(subset_df.columns))


# =========================
# 6. Detect core text columns
# =========================
infer_df = infer_df.copy()
subset_df = subset_df.copy()

infer_df["generated_report"] , detected_gen_col = build_generated_report(infer_df)
subset_df["reference_report"], detected_ref_col = build_reference_report(subset_df)

print_block("Detected generated column", detected_gen_col)
print_block("Detected reference column", detected_ref_col)


# =========================
# 7. Add join keys
# =========================
infer_df, infer_meta = add_key_columns(infer_df, prefix="infer_")
subset_df, subset_meta = add_key_columns(subset_df, prefix="subset_")

print_block("Inference key meta", infer_meta)
print_block("Subset key meta", subset_meta)


# =========================
# 8. Add row ids
# =========================
infer_df["_infer_row_id"] = np.arange(len(infer_df))
subset_df["_subset_row_id"] = np.arange(len(subset_df))


# =========================
# 9. Hierarchical matching
#    priority: study_id > dicom_id > image_stem
# =========================
used_infer = set()
used_subset = set()
matched_parts = []

def hierarchical_match(left_df, right_df, left_key, right_key, match_source):
    left = left_df[~left_df["_infer_row_id"].isin(used_infer)].copy()
    right = right_df[~right_df["_subset_row_id"].isin(used_subset)].copy()

    left = left[left[left_key].notna() & (left[left_key] != "")]
    right = right[right[right_key].notna() & (right[right_key] != "")]

    if left.empty or right.empty:
        return pd.DataFrame()

    merged = left.merge(
        right,
        left_on=left_key,
        right_on=right_key,
        how="inner",
        suffixes=("_infer", "_subset")
    )

    if merged.empty:
        return merged

    merged = merged.drop_duplicates(subset=["_infer_row_id", "_subset_row_id"]).copy()
    merged["match_source"] = match_source
    return merged

matching_plan = [
    ("infer_id_key", "subset_id_key", "id"),
    ("infer_study_id_key", "subset_study_id_key", "study_id"),
    ("infer_dicom_id_key", "subset_dicom_id_key", "dicom_id"),
    ("infer_image_stem_key", "subset_image_stem_key", "image_stem"),
]

for left_key, right_key, source in matching_plan:
    part = hierarchical_match(infer_df, subset_df, left_key, right_key, source)
    print_block(f"Matches via {source}", len(part))
    if not part.empty:
        used_infer.update(part["_infer_row_id"].tolist())
        used_subset.update(part["_subset_row_id"].tolist())
        matched_parts.append(part)

if not matched_parts:
    raise ValueError("No aligned rows found. Check IDs and column names.")

aligned = pd.concat(matched_parts, ignore_index=True)
print_block("Total aligned rows", len(aligned))


# =========================
# 10. Build clean aligned table
# =========================
aligned = aligned.copy()

def col_if_exists(df, col):
    return col if col in df.columns else None

id_cols = [
    "id", "id_infer", "id_subset",
    "sample_id", "sample_id_infer", "sample_id_subset"
]
study_cols = [
    "study_id", "study_id_infer", "study_id_subset",
    "StudyID", "StudyID_infer", "StudyID_subset"
]
dicom_cols = [
    "dicom_id", "dicom_id_infer", "dicom_id_subset",
    "DicomID", "DicomID_infer", "DicomID_subset"
]
image_cols = [
    "image_path", "image_path_infer", "image_path_subset",
    "img_path", "img_path_infer", "img_path_subset",
    "path", "path_infer", "path_subset",
    "file_path", "file_path_infer", "file_path_subset",
    "image", "image_infer", "image_subset"
]

rows = []
for _, row in aligned.iterrows():
    out = {}

    out["id"] = choose_best_value(row, id_cols)
    out["study_id"] = choose_best_value(row, study_cols)
    out["dicom_id"] = choose_best_value(row, dicom_cols)
    out["image_path"] = choose_best_value(row, image_cols)
    out["image_stem"] = basename_without_ext(out["image_path"]) if out["image_path"] else None

    out["generated_report"] = normalize_text(row["generated_report"])
    out["reference_report"] = normalize_text(row["reference_report"])
    out["match_source"] = row["match_source"]

    out["infer_row_id"] = row["_infer_row_id"]
    out["subset_row_id"] = row["_subset_row_id"]

    rows.append(out)

result = pd.DataFrame(rows)
result["case_id"] = result.apply(safe_case_id, axis=1)

result = result[
    ["case_id", "id", "study_id", "dicom_id", "image_path", "image_stem",
     "generated_report", "reference_report", "match_source",
     "infer_row_id", "subset_row_id"]
].copy()

result = result.drop_duplicates(subset=["case_id"]).reset_index(drop=True)

print_block("Aligned result shape", result.shape)
display(result.head())


# =========================
# 11. Diagnostics
# =========================
empty_mask = (
    result["generated_report"].eq("") |
    result["reference_report"].eq("")
)

empty_rows = result[empty_mask].copy()
valid_rows = result[~empty_mask].copy()

print_block("Rows with empty report text", len(empty_rows))
print_block("Rows valid for RadGraph", len(valid_rows))
print_block("Match source counts", valid_rows["match_source"].value_counts(dropna=False))

empty_rows.to_csv(EMPTY_REPORTS_OUT, index=False)


# unmatched inference rows
unmatched_infer = infer_df[~infer_df["_infer_row_id"].isin(result["infer_row_id"])].copy()
unmatched_subset = subset_df[~subset_df["_subset_row_id"].isin(result["subset_row_id"])].copy()

unmatched_infer.to_csv(UNMATCHED_INFER_OUT, index=False)
unmatched_subset.to_csv(UNMATCHED_SUBSET_OUT, index=False)

print_block("Unmatched inference rows", len(unmatched_infer))
print_block("Unmatched subset rows", len(unmatched_subset))


# =========================
# 12. Save aligned master files
# =========================
result.to_csv(ALIGNED_CSV_OUT, index=False)

with open(ALIGNED_JSONL_OUT, "w", encoding="utf-8") as f:
    for _, row in result.iterrows():
        f.write(json.dumps(row.to_dict(), ensure_ascii=False) + "\n")

print_block("Saved aligned CSV", ALIGNED_CSV_OUT)
print_block("Saved aligned JSONL", ALIGNED_JSONL_OUT)


# =========================
# 13. Build minimal RadGraph input
# =========================
# Keep only clean rows with paired texts
radgraph_df = valid_rows.copy()

# Add optional aliases that many downstream scripts expect
radgraph_df["hypothesis"] = radgraph_df["generated_report"]
radgraph_df["reference"] = radgraph_df["reference_report"]

radgraph_export = radgraph_df[
    ["case_id", "study_id", "dicom_id", "image_path",
     "generated_report", "reference_report", "hypothesis", "reference", "match_source"]
].copy()

radgraph_export.to_csv(RADGRAPH_INPUT_CSV_OUT, index=False)

with open(RADGRAPH_INPUT_JSONL_OUT, "w", encoding="utf-8") as f:
    for _, row in radgraph_export.iterrows():
        f.write(json.dumps(row.to_dict(), ensure_ascii=False) + "\n")

print_block("Saved RadGraph CSV", RADGRAPH_INPUT_CSV_OUT)
print_block("Saved RadGraph JSONL", RADGRAPH_INPUT_JSONL_OUT)


# =========================
# 14. Quick sanity preview
# =========================
preview_n = min(5, len(radgraph_export))
if preview_n > 0:
    display(
        radgraph_export.sample(preview_n, random_state=42)[
            ["case_id", "match_source", "generated_report", "reference_report"]
        ]
    )
else:
    print("No valid rows available for preview.")



=============== Inference shape ===============
(452, 4)

=============== Subset shape ===============
(1500, 11)

=============== Inference columns ===============
['id', 'query', 'reference', 'prediction']

=============== Subset columns ===============
['id', 'image', 'generate_method', 'reason', 'impression', 'indication', 'history', 'view', 'orientation', 'chexpert_labels', 'conversations']

=============== Detected generated column ===============
prediction

=============== Detected reference column ===============
impression

=============== Inference key meta ===============
{'id_col': 'id', 'study_col': None, 'dicom_col': None, 'image_col': None}

=============== Subset key meta ===============
{'id_col': 'id', 'study_col': None, 'dicom_col': None, 'image_col': 'image'}

=============== Matches via id ===============
1514

=============== Matches via study_id ===============
0

=============== Matches via dicom_id ===============
0

=============== Matches via image_stem ===

,case_id,id,study_id,dicom_id,image_path,image_stem,generated_report,reference_report,match_source,infer_row_id,subset_row_id
0,1004616657379357,1004616657379357,None,None,mimic/p10/p10046166/s57379357/6e511483-c7e1601...,6e511483-c7e1601c-76890b2f-b0c6b55d-e53bcbf6,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,id,0,14
1,1004616657977208,1004616657977208,None,None,mimic/p10/p10046166/s57977208/e2856783-ffa5ec2...,e2856783-ffa5ec26-043b0303-21aeddc6-b11b2876,"In comparison with the study of ___, there is ...",,id,1,18
2,1026887750042142,1026887750042142,None,None,mimic/p10/p10268877/s50042142/4c3c1335-0fce9b1...,4c3c1335-0fce9b11-027c582b-a0ed8d89-ca614d90,Endotracheal tube tip terminates approximately...,,id,2,21
3,1026887750239281,1026887750239281,None,None,mimic/p10/p10268877/s50239281/0c69d156-6f5f3a8...,0c69d156-6f5f3a89-7d361367-57f8c979-583ef198,Single AP upright portable view of the chest w...,1. Left PICC tip appears to terminate in the d...,id,3,24
4,1026887751513702,1026887751513702,None,None,mimic/p10/p10268877/s51513702/053e0fdd-17dbee8...,053e0fdd-17dbee89-17885e49-08249a30-7f829c9c,Single AP upright portable view of the chest w...,No definite acute cardiopulmonary process. Enl...,id,4,28



=============== Rows with empty report text ===============
132

=============== Rows valid for RadGraph ===============
272

=============== Match source counts ===============
match_source
id    272
Name: count, dtype: int64

=============== Unmatched inference rows ===============
48

=============== Unmatched subset rows ===============
1096

=============== Saved aligned CSV ===============
/workspace/thesis/outputs/aligned_inference_1500_subset.csv

=============== Saved aligned JSONL ===============
/workspace/thesis/outputs/aligned_inference_1500_subset.jsonl

=============== Saved RadGraph CSV ===============
/workspace/thesis/outputs/radgraph_input_1500_subset.csv

=============== Saved RadGraph JSONL ===============
/workspace/thesis/outputs/radgraph_input_1500_subset.jsonl


,case_id,match_source,generated_report,reference_report
51,1093360951002383,id,The heart size is normal. The mediastinal and ...,Bilateral upper lobe scarring without evidence...
191,1151210456889771,id,The heart is mild to moderately enlarged. The ...,"Moderate pulmonary edema and cardiomegaly, wit..."
120,1156580359027235,id,"In comparison with the study of ___, the patie...","Bilateral lower lung atelectasis, mild-to-mode..."
214,1105227353537165,id,The heart is mildly enlarged. The mediastinal ...,No definite evidence for congestive heart fail...
317,1071547759170987,id,The endotracheal tube terminates 4.5 cm above ...,Mild acute congestive heart failure.
